In [2]:
pip install transformers torch

In [8]:
import torch
from torch.utils.data import Dataset
from transformers import (
    GPT2Tokenizer,
    GPT2LMHeadModel,
    DataCollatorForLanguageModeling,
    Trainer,
    TrainingArguments
)

# Create custom dataset file
with open("custom_text.txt", "w") as file:
    file.write("""
    Machine learning is a branch of artificial intelligence.
    Deep learning uses neural networks.
    NLP helps computers understand human language.
    Artificial intelligence is changing the world.
    """)

tokenizer = GPT2Tokenizer.from_pretrained("gpt2")
tokenizer.pad_token = tokenizer.eos_token

model = GPT2LMHeadModel.from_pretrained("gpt2")
model.config.pad_token_id = tokenizer.eos_token_id

# Define a custom Dataset with explicit labels for trainer loss calculation
class SimpleTextDataset(Dataset):
    def __init__(self, file_path, tokenizer, block_size=32):
        with open(file_path, "r", encoding="utf-8") as f:
            lines = [line.strip() for line in f.readlines() if line.strip()]

        self.examples = []
        for line in lines:
            tokenized = tokenizer(
                line,
                truncation=True,
                max_length=block_size,
                padding="max_length"
            )
            input_ids = torch.tensor(tokenized["input_ids"], dtype=torch.long)
            attention_mask = torch.tensor(tokenized["attention_mask"], dtype=torch.long)
            # GPT-2 Trainer calculates language modeling loss using input_ids as labels
            labels = input_ids.clone()

            self.examples.append({
                "input_ids": input_ids,
                "attention_mask": attention_mask,
                "labels": labels
            })

    def __len__(self):
        return len(self.examples)

    def __getitem__(self, idx):
        return self.examples[idx]

dataset = SimpleTextDataset(
    file_path="custom_text.txt",
    tokenizer=tokenizer,
    block_size=32
)

data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False
)

training_args = TrainingArguments(
    output_dir="./gpt2_custom",
    num_train_epochs=3,
    per_device_train_batch_size=2,
    save_strategy="no",
    report_to="none"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=dataset,
    data_collator=data_collator
)

trainer.train()

# Generate text\nprompt = "Machine learning"

inputs = tokenizer(prompt, return_tensors="pt")

output = model.generate(
    **inputs,
    max_new_tokens=40,
    do_sample=True,
    temperature=0.7,
    pad_token_id=tokenizer.eos_token_id
)

print(tokenizer.decode(output[0], skip_special_tokens=True))

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

Step,Training Loss


NameError: name 'prompt' is not defined